# 01 — OLLAMA Setup & Connectivity Check

**Vai trò:** Model Engineer · **Task:** S1-ME-02 (Yêu cầu 9.1, 9.4)

Notebook này kiểm tra OLLAMA server cục bộ đã sẵn sàng chưa và những model nào đã được pull về — bước khởi đầu bắt buộc trước khi làm việc với embedding (Sprint 2) và sinh câu trả lời (Sprint 3), vì toàn bộ hệ thống chỉ dùng LLM **chạy cục bộ** qua OLLAMA, không phụ thuộc dịch vụ đám mây.

In [29]:
import sys 
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
  sys.path.insert(0, str(PROJECT_ROOT))

from src.generation.llm_client import OllamaClient
from config.settings import AppConfig

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/phamhaithien/Documents/AIO_VN/Workspace_RAG_PDF/AI-Research-Assistant-with-RAG


## 1. Khởi tạo OllamaClient

Mặc định kết nối tới `http://localhost:11434` với model `vicuna:7b-v1.5-q5_1` (xem `config/settings.py` — `OllamaConfig`).

In [30]:
cfg = AppConfig.from_env()

print("Cấu hình OLLAMA (AppConfig):")
print(f"  base_url                : {cfg.ollama.base_url}")
print(f"  default_llm_model       : {cfg.ollama.default_llm_model}")
print(f"  default_embedding_model : {cfg.ollama.default_embedding_model}")
print(f"  timeout_seconds         : {cfg.ollama.timeout_seconds}")

Cấu hình OLLAMA (AppConfig):
  base_url                : http://localhost:11434
  default_llm_model       : vicuna:7b-v1.5-q5_1
  default_embedding_model : bge-m3:latest
  timeout_seconds         : 120


## 2. Kiểm tra server khả dụng — `is_available()`

Theo Yêu cầu 5.2, `is_available()` **không bao giờ ném exception** — luôn trả về `True`/`False`, kể cả khi server chưa chạy hoặc không thể kết nối. Vì vậy cell dưới đây luôn chạy xong an toàn dù OLLAMA đã khởi động hay chưa.

In [31]:
available = client.is_available()
print(f"OLLAMA server khả dụng tại {client.base_url}: {available}")

if not available:
    print(
        "\n⚠️  Server chưa sẵn sàng. Hãy mở một terminal khác và chạy:\n"
        "    ollama serve\n"
        "    ollama pull llama3\n"
        "    ollama pull nomic-embed-text\n"
        "rồi chạy lại notebook này từ đầu."
    )

OLLAMA server khả dụng tại http://localhost:11434: True


## 3. Danh sách model đã pull — `list_models()`

Trả về danh sách rỗng (không ném exception) nếu server không khả dụng hoặc chưa pull model nào (Yêu cầu 5.4).

In [32]:
models = client.list_models()
if models:
    print("Models đã pull về OLLAMA:")
    for name in models:
        print(f"  - {name}")
else:
    print("Chưa có model nào hoặc server không khả dụng — xem hướng dẫn ở mục 2.")

Models đã pull về OLLAMA:
  - bge-m3:latest
  - vicuna:7b-v1.5-q5_1


## 4. Tổng kết

- `is_available()` cho biết OLLAMA server cục bộ đã sẵn sàng phục vụ embedding/generation chưa.
- `list_models()` cho biết những model nào (vd. `llama3`, `nomic-embed-text`) đã có sẵn để dùng ở các sprint tiếp theo: Sprint 2 cần `nomic-embed-text` cho embedding, Sprint 3 cần `llama3` (hoặc model tương đương) cho sinh câu trả lời.
- Notebook chạy được từ đầu đến cuối **không phát sinh exception**, dù server đang chạy hay chưa — đúng kỳ vọng "notebook phải chạy được từ đầu đến cuối mà không gặp lỗi ngoại lệ chưa xử lý" trong CLAUDE.md.